In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

#Data
df = pd.read_csv("../../data/geocoded_data/new_scores.csv")

# Keep only valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print(df.shape)
df.head()


C:\Users\shrey\AppData\Local\Temp\ipykernel_44772\2853862544.py:6: DtypeWarning: Columns (3,4,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/geocoded_data/new_scores.csv")


(442395, 32)


,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,domain_weight,issue_weight,severity_score,expected_resolution,resolution_factor,q10,q25,q75,q90,resolution_score
0,2021-03-02 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:56,NaN,Resolved,CS0063095,2021-09-13 11:28:56,4677.982222,...,2,2,4,10.614444,440.718518,0.905370,0.911311,5.768855,88.451246,1
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:21,NaN,Resolved,CS0060088,2021-07-15 12:54:21,3215.405833,...,5,1,5,9.782083,328.703582,0.979029,0.984112,2.391575,32.619642,1
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:34,NaN,Resolved,CS0026984,2022-05-26 09:54:34,10772.409444,...,5,1,5,9.782083,1101.238773,0.979029,0.984112,2.391575,32.619642,1
3,2021-03-09 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:37,NaN,Resolved,CS0001758,2021-04-14 15:24:37,865.910278,...,4,3,12,51.054722,16.960435,0.226881,0.507299,13.972149,336.648912,2
4,2021-03-11 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-06-03 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283333,...,4,1,4,93.250000,397.686685,0.104471,0.283426,9.242181,63.591008,1


In [2]:
#Convert to GeoDataFrame + Load Block Groups

geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(gdf.crs)


EPSG:4326


In [3]:
import requests
import zipfile
import os

# URL for Georgia Block Groups
url = "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_13_bg.zip"

# Local paths
zip_path = "tl_2020_13_bg.zip"
extract_path = "tl_2020_13_bg"

# Download
response = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(response.content)

print("Download complete")

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

Download complete
Extraction complete


In [4]:
import geopandas as gpd

block_groups = gpd.read_file("tl_2020_13_bg/tl_2020_13_bg.shp")

print(block_groups.shape)
# block_groups.head()
block_groups = block_groups.to_crs(gdf.crs)

(7446, 13)


In [5]:
block_groups.to_file("block_groups.gpkg", driver="GPKG")
block_groups = gpd.read_file("block_groups.gpkg")

In [6]:
joined = gpd.sjoin(
    gdf,
    block_groups,
    how="left",
    predicate="within"
)

In [7]:
print(joined.shape)
joined.head()

(442395, 46)


,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,TRACTCE,BLKGRPCE,GEOID,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON
0,2021-03-02 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:56,NaN,Resolved,CS0063095,2021-09-13 11:28:56,4677.982222,...,010211,4,131210102114,Block Group 4,G5030,S,1510335,30512,+33.8846025,-084.3942389
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:21,NaN,Resolved,CS0060088,2021-07-15 12:54:21,3215.405833,...,000400,1,131210004001,Block Group 1,G5030,S,570020,0,+33.7909887,-084.3841000
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:34,NaN,Resolved,CS0026984,2022-05-26 09:54:34,10772.409444,...,009900,2,131210099002,Block Group 2,G5030,S,2674203,0,+33.8491900,-084.4004196
3,2021-03-09 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:37,NaN,Resolved,CS0001758,2021-04-14 15:24:37,865.910278,...,007900,1,131210079001,Block Group 1,G5030,S,3109094,0,+33.7311273,-084.4805695
4,2021-03-11 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-06-03 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283333,...,009606,1,131210096061,Block Group 1,G5030,S,1648224,0,+33.8407847,-084.3644458


In [8]:
joined["GEOID"].isna().sum()

np.int64(0)

In [9]:
joined[["lat", "lng", "GEOID"]].head()

,lat,lng,GEOID
0,33.877556,-84.388201,131210102114
1,33.795376,-84.386323,131210004001
2,33.848637,-84.390632,131210099002
3,33.723403,-84.475014,131210079001
4,33.833415,-84.367372,131210096061


In [10]:
joined.head()

,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,TRACTCE,BLKGRPCE,GEOID,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON
0,2021-03-02 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:56,NaN,Resolved,CS0063095,2021-09-13 11:28:56,4677.982222,...,010211,4,131210102114,Block Group 4,G5030,S,1510335,30512,+33.8846025,-084.3942389
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:21,NaN,Resolved,CS0060088,2021-07-15 12:54:21,3215.405833,...,000400,1,131210004001,Block Group 1,G5030,S,570020,0,+33.7909887,-084.3841000
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:34,NaN,Resolved,CS0026984,2022-05-26 09:54:34,10772.409444,...,009900,2,131210099002,Block Group 2,G5030,S,2674203,0,+33.8491900,-084.4004196
3,2021-03-09 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:37,NaN,Resolved,CS0001758,2021-04-14 15:24:37,865.910278,...,007900,1,131210079001,Block Group 1,G5030,S,3109094,0,+33.7311273,-084.4805695
4,2021-03-11 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-06-03 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283333,...,009606,1,131210096061,Block Group 1,G5030,S,1648224,0,+33.8407847,-084.3644458


In [11]:
bg_year_agg = joined.groupby(["GEOID", "year"]).agg({
    "Number": "count",
    "severity_score": "mean",
    "resolution_score": "mean",
    "resolution_time_hours": "mean"
}).reset_index()

bg_year_agg = bg_year_agg.rename(columns={
    "Number": "complaint_count",
    "severity_score": "avg_severity",
    "resolution_score": "avg_resolution_score",
    "resolution_time_hours": "avg_resolution_time"
})

In [12]:
issue_counts = (
    joined
    .groupby(["GEOID", "year", "issue_type"])
    .size()
    .reset_index(name="count")
)

issue_dict = (
    issue_counts
    .groupby(["GEOID", "year"])
    .apply(lambda x: dict(zip(x["issue_type"], x["count"])))
    .reset_index(name="issue_type_counts")
)

domain_counts = (
    joined
    .groupby(["GEOID", "year", "domain"])
    .size()
    .reset_index(name="count")
)


domain_dict = (
    domain_counts
    .groupby(["GEOID", "year"])
    .apply(lambda x: dict(zip(x["domain"], x["count"])))
    .reset_index(name="domain_counts")
)

bg_year_agg = bg_year_agg.merge(issue_dict, on=["GEOID", "year"], how="left")
bg_year_agg = bg_year_agg.merge(domain_dict, on=["GEOID", "year"], how="left")


bg_year_agg["issue_type_counts"] = bg_year_agg["issue_type_counts"].apply(
    lambda x: x if isinstance(x, dict) else {}
)

bg_year_agg["domain_counts"] = bg_year_agg["domain_counts"].apply(
    lambda x: x if isinstance(x, dict) else {}
)

C:\Users\shrey\AppData\Local\Temp\ipykernel_44772\2594942233.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x["issue_type"], x["count"])))
C:\Users\shrey\AppData\Local\Temp\ipykernel_44772\2594942233.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x["domain"], x["count"])))


In [13]:
print(bg_year_agg.shape)
bg_year_agg.head()

(3094, 8)


,GEOID,year,complaint_count,avg_severity,avg_resolution_score,avg_resolution_time,issue_type_counts,domain_counts
0,130630402021,2021,5,5.000000,3.000000,10.597944,{'Information / Other': 5},{'Water & Sewer': 5}
1,130630402021,2022,2,5.000000,3.000000,10.130556,{'Service Request': 2},"{'Administrative / Account Services': 1, 'Wast..."
2,130630402021,2023,5,8.800000,3.400000,13.404556,"{'Hazard / Emergency': 1, 'Information / Other...","{'Public Safety': 1, 'Waste Management': 2, 'W..."
3,130630402021,2024,7,2.714286,3.000000,19.964563,"{'Information / Other': 6, 'Service Request': 1}","{'Administrative / Account Services': 1, 'Code..."
4,130630402021,2025,7,10.142857,3.714286,35.814286,"{'Hazard / Emergency': 1, 'Information / Other...","{'Administrative / Account Services': 2, 'Code..."


In [14]:
block_groups_proj = block_groups.to_crs(epsg=3857)

block_groups_proj["area_sq_km"] = block_groups_proj.geometry.area / 1e6

In [15]:
bg_year_agg = bg_year_agg.merge(
    block_groups_proj[["GEOID", "area_sq_km"]],
    on="GEOID",
    how="left"
)

In [16]:
bg_year_agg["complaint_density"] = (
    bg_year_agg["complaint_count"] / bg_year_agg["area_sq_km"]
)

In [17]:
bg_year_agg.describe()

,year,complaint_count,avg_severity,avg_resolution_score,avg_resolution_time,area_sq_km,complaint_density
count,3094.000000,3094.000000,3094.000000,3094.000000,3094.000000,3094.000000,3094.000000
mean,2023.107628,142.984809,7.865278,3.027514,1216.276700,1.568827,150.315637
std,1.389904,157.813909,3.195604,0.563321,2191.201220,1.836232,161.230112
min,2021.000000,1.000000,1.000000,1.000000,0.033333,0.030608,0.046432
25%,2022.000000,7.000000,6.612157,2.753088,108.972865,0.578932,4.974425
50%,2023.000000,88.000000,7.693399,3.000000,295.433163,1.015693,108.007852
75%,2024.000000,237.000000,8.886804,3.284702,1193.960972,1.840550,236.374587
max,2025.000000,937.000000,25.000000,5.000000,23698.916667,21.537093,975.334857


In [18]:
bg_year_agg.columns

Index(['GEOID', 'year', 'complaint_count', 'avg_severity',
       'avg_resolution_score', 'avg_resolution_time', 'issue_type_counts',
       'domain_counts', 'area_sq_km', 'complaint_density'],
      dtype='object')

In [19]:
# #Plot
# map_df = block_groups.merge(
#     bg_year_agg,   # or final_df if you want ACS features too
#     on="GEOID",
#     how="left"
# )

# import matplotlib.pyplot as plt

# years = sorted(map_df["year"].dropna().unique())

# for yr in years:
#     fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
#     data = map_df[map_df["year"] == yr]
    
#     data.plot(
#         column="avg_severity",   # change this to other metrics later
#         cmap="OrRd",
#         legend=True,
#         ax=ax
#     )
    
#     ax.set_title(f"Severity - {yr}")
#     ax.axis("off")
    
#     plt.show()

In [20]:
# Extract county FIPS from GEOID
joined["county_fips"] = joined["GEOID"].str[2:5]

# Get unique counties
counties = joined["county_fips"].dropna().unique()

# Ensure proper formatting (important for API)
counties = [str(c).zfill(3) for c in counties]

print("Counties in dataset:", counties)
print("Number of counties:", len(counties))

Counties in dataset: ['121', '089', '067', '063', '151']
Number of counties: 5


In [21]:
from census import Census
import pandas as pd

c = Census("fa7883b213fcc5047737a09467ed916145ae32fa")

variables = (
    "B19013_001E",
    "B01003_001E",
    "B25003_003E",
    "B25003_001E",
    "C17002_002E",

    "B02001_001E",  # total race population
    "B02001_002E",  # white
    "B02001_003E",  # black
    "B02001_005E"   # asian

)

years = [2021, 2022, 2023]  # ACS available years

acs_all = []

for year in years:
    print(f"Fetching ACS {year}")
    
    for county in counties:
        data = c.acs5.state_county_blockgroup(
            variables,
            state_fips="13",
            county_fips=county,
            blockgroup="*",
            year=year
        )
        
        # attach year to each row
        for row in data:
            row["acs_year"] = year
        
        acs_all.extend(data)

acs_df = pd.DataFrame(acs_all)
print(acs_df.columns)

Fetching ACS 2021
Fetching ACS 2022
Fetching ACS 2023
Index(['B19013_001E', 'B01003_001E', 'B25003_003E', 'B25003_001E',
       'C17002_002E', 'B02001_001E', 'B02001_002E', 'B02001_003E',
       'B02001_005E', 'state', 'county', 'tract', 'block group', 'acs_year'],
      dtype='object')


In [22]:
acs_df["GEOID"] = (
    acs_df["state"] +
    acs_df["county"] +
    acs_df["tract"] +
    acs_df["block group"]
)

print(acs_df.shape)

(6702, 15)


In [23]:
# Rename important columns
acs_df = acs_df.rename(columns={
    "B19013_001E": "median_income",
    "B01003_001E": "population",
    "C17002_002E": "poverty_rate",

    "B02001_001E": "race_total",
    "B02001_002E": "white_pop",
    "B02001_003E": "black_pop",
    "B02001_005E": "asian_pop"
})

print("Before cleaning:", acs_df.shape)

print("Missing values:")
print(acs_df[[
    "population",
    "median_income",
    "poverty_rate"
]].isna().sum())

Before cleaning: (6702, 15)
Missing values:
population       0
median_income    0
poverty_rate     0
dtype: int64


In [24]:
print(acs_df.columns)

Index(['median_income', 'population', 'B25003_003E', 'B25003_001E',
       'poverty_rate', 'race_total', 'white_pop', 'black_pop', 'asian_pop',
       'state', 'county', 'tract', 'block group', 'acs_year', 'GEOID'],
      dtype='object')


In [25]:
# Convert to numeric
cols_to_numeric = [
    "median_income",
    "population",
    "B25003_003E",
    "B25003_001E",
    "poverty_rate",
    "race_total",
    "white_pop",
    "black_pop",
    "asian_pop"
]

for col in cols_to_numeric:
    acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")

# Keep only valid population
acs_df = acs_df.dropna(subset=["population"])
acs_df = acs_df[acs_df["population"] > 0]

acs_df = acs_df[acs_df["median_income"] > 0]

acs_df = acs_df.replace(-666666666, pd.NA)

# Derived feature
acs_df["pct_renters"] = (
    acs_df["B25003_003E"] / acs_df["B25003_001E"]
)

In [26]:
acs_df["pct_white"] = acs_df["white_pop"] / acs_df["race_total"]
acs_df["pct_black"] = acs_df["black_pop"] / acs_df["race_total"]
acs_df["pct_asian"] = acs_df["asian_pop"] / acs_df["race_total"]

In [27]:
acs_df.shape

(6063, 19)

In [28]:
def map_acs_year(y):
    if y <= 2023:
        return y
    else:
        return 2023 

bg_year_agg["acs_year"] = bg_year_agg["year"].apply(map_acs_year)

acs_df = acs_df[acs_df["GEOID"].isin(bg_year_agg["GEOID"])]
print("Filtered ACS shape:", acs_df.shape)

Filtered ACS shape: (2001, 19)


In [29]:
final_df = bg_year_agg.merge(
    acs_df,
    on=["GEOID", "acs_year"],
    how="left"
)

In [30]:
final_df["complaints_per_1000"] = (
    final_df["complaint_count"] / final_df["population"] * 1000
)

In [31]:
final_df = final_df.sort_values(["GEOID", "year"])

# final_df["income_change"] = final_df.groupby("GEOID")["median_income"].diff()
# final_df["severity_change"] = final_df.groupby("GEOID")["avg_severity"].diff()

In [32]:
final_df

,GEOID,year,complaint_count,avg_severity,avg_resolution_score,avg_resolution_time,issue_type_counts,domain_counts,area_sq_km,complaint_density,...,asian_pop,state,county,tract,block group,pct_renters,pct_white,pct_black,pct_asian,complaints_per_1000
0,130630402021,2021,5,5.000000,3.000000,10.597944,{'Information / Other': 5},{'Water & Sewer': 5},12.958286,0.385854,...,51.0,13,063,040202,1,0.776386,0.038308,0.846768,0.040702,3.990423
1,130630402021,2022,2,5.000000,3.000000,10.130556,{'Service Request': 2},"{'Administrative / Account Services': 1, 'Wast...",12.958286,0.154341,...,82.0,13,063,040202,1,0.775618,0.010554,0.764292,0.072120,1.759015
2,130630402021,2023,5,8.800000,3.400000,13.404556,"{'Hazard / Emergency': 1, 'Information / Other...","{'Public Safety': 1, 'Waste Management': 2, 'W...",12.958286,0.385854,...,71.0,13,063,040202,1,0.777096,0.002732,0.707650,0.064663,4.553734
3,130630402021,2024,7,2.714286,3.000000,19.964563,"{'Information / Other': 6, 'Service Request': 1}","{'Administrative / Account Services': 1, 'Code...",12.958286,0.540195,...,71.0,13,063,040202,1,0.777096,0.002732,0.707650,0.064663,6.375228
4,130630402021,2025,7,10.142857,3.714286,35.814286,"{'Hazard / Emergency': 1, 'Information / Other...","{'Administrative / Account Services': 2, 'Code...",12.958286,0.540195,...,71.0,13,063,040202,1,0.777096,0.002732,0.707650,0.064663,6.375228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3089,131219800001,2022,1,20.000000,2.000000,1109.776111,{'Infrastructure Damage': 1},{'Public Safety': 1},3.947301,0.253338,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3090,131219800001,2023,11,12.272727,3.272727,167.003232,"{'Hazard / Emergency': 2, 'Information / Other...","{'Licensing & Permits': 3, 'Public Safety': 3,...",3.947301,2.786714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3091,131219800001,2024,12,8.166667,2.916667,453.674861,"{'Information / Other': 5, 'Infrastructure Dam...","{'Administrative / Account Services': 1, 'Lice...",3.947301,3.040052,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3092,131219800001,2025,10,11.600000,3.800000,94.896667,"{'Information / Other': 2, 'Infrastructure Dam...","{'Licensing & Permits': 1, 'Road & Infrastruct...",3.947301,2.533377,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
analysis_df = final_df.dropna(subset=[
    "complaints_per_1000"
])

In [34]:
analysis_df.to_csv("../../data/geocoded_data/composite_scores.csv", index=False)

In [35]:
analysis_df.columns

Index(['GEOID', 'year', 'complaint_count', 'avg_severity',
       'avg_resolution_score', 'avg_resolution_time', 'issue_type_counts',
       'domain_counts', 'area_sq_km', 'complaint_density', 'acs_year',
       'median_income', 'population', 'B25003_003E', 'B25003_001E',
       'poverty_rate', 'race_total', 'white_pop', 'black_pop', 'asian_pop',
       'state', 'county', 'tract', 'block group', 'pct_renters', 'pct_white',
       'pct_black', 'pct_asian', 'complaints_per_1000'],
      dtype='object')

In [36]:
print(block_groups["GEOID"].head())
print(analysis_df["GEOID"].head())
print(len(block_groups["GEOID"].iloc[0]))
print(len(analysis_df["GEOID"].iloc[0]))

0    131759505003
1    131759501002
2    132099503001
3    132099502003
4    132099501001
Name: GEOID, dtype: object
0    130630402021
1    130630402021
2    130630402021
3    130630402021
4    130630402021
Name: GEOID, dtype: object
12
12


In [37]:
valid_geoids = analysis_df["GEOID"].unique()

final_gdf = block_groups[
    block_groups["GEOID"].isin(valid_geoids)
].merge(
    analysis_df,
    on="GEOID",
    how="left",
    indicator=True
)

print(final_gdf["_merge"].value_counts())

_merge
both          2621
left_only        0
right_only       0
Name: count, dtype: int64


In [39]:
cols_to_check = [
    "complaints_per_1000",
    "avg_severity",
    "avg_resolution_score",
    "median_income"]

print(final_gdf[cols_to_check].isna().sum())

complaints_per_1000     0
avg_severity            0
avg_resolution_score    0
median_income           0
dtype: int64


In [ ]:
years = sorted(final_gdf["year"].dropna().unique())
print(years)

In [ ]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

import os

os.makedirs("geojson_outputs", exist_ok=True)

for year in years:
    df_year = final_gdf[final_gdf["year"] == year].copy()
    
    # ensure one row per GEOID
    df_year = df_year.drop_duplicates(subset=["GEOID"])
    
    # fill missing values
    df_year = df_year.fillna(0)
    
    # optional: compute final score
    df_year["final_score"] = (
    0.4 * normalize(df_year["complaints_per_1000"]) +
    0.3 * normalize(df_year["avg_severity"]) +
    0.3 * normalize(df_year["avg_resolution_score"])
) * 100
    
    # save file
    output_path = f"geojson_outputs/map_{year}.geojson"
    df_year.to_file(output_path, driver="GeoJSON")
    
    print(f"Saved {output_path}")

In [ ]:
import geopandas as gpd
test = gpd.read_file("geojson_outputs/map_2023.geojson")
print(test.head())

In [ ]:
corr = analysis_df[[
    "complaints_per_1000",
    "avg_severity",
    "avg_resolution_time",
    "median_income",
    "poverty_rate",
    "pct_renters",
    "pct_black"
]].corr()

print(corr)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(corr, annot=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
import statsmodels.api as sm

X = analysis_df[[
    "median_income",
    "poverty_rate",
    "pct_renters"
]]

y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
map_df = block_groups.merge(
    final_df,
    on="GEOID",
    how="left"
)

map_df.plot(
    column="complaints_per_1000",
    legend=True,
    figsize=(10, 8)
)
plt.title("Complaint Density by Block Group")
plt.show()

In [ ]:
map_df.plot(
    column="median_income",
    legend=True,
    figsize=(10, 8)
)
plt.title("Median Income by Block Group")
plt.show()

In [ ]:
# Regression

In [ ]:
import statsmodels.api as sm

X = analysis_df[[
    "median_income",
    "poverty_rate",
    "pct_renters"
]]

y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
analysis_df["income_k"] = analysis_df["median_income"] / 1000

In [ ]:
X = analysis_df[["income_k", "pct_renters", "pct_black"]]
y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["pct_renters", "pct_black"]]
y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["income_k", "pct_renters", "pct_black"]]
y = analysis_df["avg_severity"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["income_k", "pct_renters", "poverty_rate", "pct_black"]]
y = analysis_df["avg_resolution_score"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
analysis_df["race_group"] = pd.qcut(
    analysis_df["pct_black"],
    3,
    labels=["low_black", "mid_black", "high_black"]
)

In [ ]:
analysis_df["race_group"].value_counts()

In [ ]:
analysis_df.groupby("race_group")[[
    "complaints_per_1000",
    "avg_severity",
    "avg_resolution_time",
    "median_income",
    "pct_renters"
]].mean()

In [ ]:
import statsmodels.api as sm

for group in ["low_black", "mid_black", "high_black"]:
    
    subset = analysis_df[analysis_df["race_group"] == group]
    
    X = subset[["income_k", "pct_renters"]]
    y = subset["complaints_per_1000"]
    
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    print("\n====================")
    print(f"Group: {group}")
    print("====================")
    print(model.summary())

In [ ]:
############### For getting names - independent pipeline

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

#Data
df = pd.read_csv("../../data/geocoded_data/new_scores.csv")

# Keep only valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print(df.shape)
df.head()

#Convert to GeoDataFrame + Load Block Groups

geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(gdf.crs)


In [ ]:
block_groups = gpd.read_file("tl_2020_13_bg/tl_2020_13_bg.shp")
block_groups = block_groups.to_crs(gdf.crs)

In [ ]:
# load neighborhoods
neighborhoods = gpd.read_file("Atlanta_Neighborhoods.geojson")

# match CRS
neighborhoods = neighborhoods.to_crs(block_groups.crs)

# keep only needed columns
neighborhoods = neighborhoods[["NAME", "geometry"]]
neighborhoods = neighborhoods.rename(columns={"NAME": "neighborhood_name"})

In [ ]:
bg_with_neighborhood = gpd.sjoin(
    block_groups,
    neighborhoods,
    how="left",
    predicate="intersects"
)

# remove duplicates
bg_with_neighborhood = bg_with_neighborhood.drop_duplicates(subset=["GEOID"])

# keep only needed columns
bg_with_neighborhood = bg_with_neighborhood[["GEOID", "neighborhood_name"]]

In [ ]:
print(bg_with_neighborhood.neighborhood_name.isna)

In [ ]:
print("block_groups CRS:", block_groups.crs)
print("neighborhoods CRS:", neighborhoods.crs)

In [ ]:
print(neighborhoods.geometry.head())

In [ ]:
print(bg_with_neighborhood["neighborhood_name"].isna().sum())

In [ ]:
bg_with_neighborhood = bg_with_neighborhood[
    bg_with_neighborhood["neighborhood_name"].notna()
]

In [ ]:
bg_with_neighborhood["GEOID"] = bg_with_neighborhood["GEOID"].astype(str)

bg_with_neighborhood.to_csv("bg_with_neighborhood.csv", index=False)

print("Saved bg_with_neighborhood.csv")

In [ ]:
import os
import json

geojson_folder = "geojson_outputs"  # change if needed

geojson_geoids = set()

for file in os.listdir(geojson_folder):
    if file.endswith(".geojson"):
        path = os.path.join(geojson_folder, file)

        with open(path) as f:
            data = json.load(f)

        for feature in data["features"]:
            geoid = feature["properties"].get("GEOID")
            if geoid:
                geojson_geoids.add(str(geoid))

print("Total GEOIDs in GeoJSON files:", len(geojson_geoids))

In [ ]:
bg_geoids = set(bg_with_neighborhood["GEOID"].astype(str))

print("Total GEOIDs in bg_with_neighborhood:", len(bg_geoids))

missing_in_bg = geojson_geoids - bg_geoids
print("GEOIDs in GeoJSON but NOT in bg_with_neighborhood:", len(missing_in_bg))

extra_in_bg = bg_geoids - geojson_geoids
print("GEOIDs in bg_with_neighborhood but NOT in GeoJSON:", len(extra_in_bg))

print("Sample missing GEOIDs:", list(missing_in_bg)[:10])
print("Sample extra GEOIDs:", list(extra_in_bg)[:10])

missing_names = bg_with_neighborhood["neighborhood_name"].isna().sum()

print("Block groups without neighborhood:", missing_names)

In [ ]:
check_geoids = [
    "131210085003","131210088022","131210080005","131210084001","131210070023",
    "131210077031","131210073013","131210055041","131210070021","131210063002",
    "130630403081","130890215021","130630402031","131210006022","131210106042",
    "130890229002","130890214161","130890214173","131210108023","130890238023"
]

In [ ]:
result = bg_with_neighborhood[
    bg_with_neighborhood["GEOID"].astype(str).isin(check_geoids)
][["GEOID", "neighborhood_name"]]

print(result)

In [ ]:
missing = result[result["neighborhood_name"].isna()]
print("Missing neighborhood mapping:")
print(missing)